# CellphoneS Product Scraper — TMDT Database Schema

## Mục tiêu
Cào toàn bộ sản phẩm điện thoại từ `https://cellphones.com.vn/mobile.html`, parse dữ liệu và map trực tiếp vào model **Product – ProductVariant** của TMDT backend.

## Database Schema mapping

| Trường scraped | → | DB field | Entity |
|---|---|---|---|
| `brand.name` | → | `brand_id` (FK) | Brand |
| `category.name` | → | `category_id` (FK) | Category |
| `name`, `slug`, `short_description`, `detail_description`, `thumbnail_url`, `warranty_months`, `sale`, `is_featured`, `status` | → | tương ứng | Product |
| `variants[].sku`, `color`, `ram_gb`, `storage_label`, `storage_gb`, `price`, `compare_at_price`, `weight_gram` | → | tương ứng | ProductVariant |
| `variants[].images` | → | `product_images` | ProductImage |
| `specs` | → | `product_specifications` | ProductSpecification |
| `rating`, `review_count` | → | tự tính trung bình | Review |

## Quy trình 2 giai đoạn
1. **Stage 1** — Listing: Exhaustively collect all product detail URLs from the category page (infinite scroll simulation).
2. **Stage 2** — Detail: For each URL, fetch the embedded JSON (`window.__NEXT_DATA__` / SSR data) and extract all fields.

In [1]:
# ─────────────────────────────────────────────
# 0. Dependencies
# ─────────────────────────────────────────────
import subprocess, sys

deps = [
    "requests>=2.31.0",
    "beautifulsoup4>=4.12.0",
    "lxml>=4.9.0",
    "playwright>=1.40.0",
    "nest-asyncio>=1.6.0",
]
for d in deps:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", d])

import nest_asyncio
nest_asyncio.apply()          # cho phép nested event loop trong Jupyter

print("All dependencies installed.")

All dependencies installed.


In [2]:
# ─────────────────────────────────────────────
# 1. Imports & Configuration
# ─────────────────────────────────────────────
import json, time, re, hashlib, unicodedata
from datetime import datetime, timezone
from typing import Optional
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup, Tag

BASE_URL   = "https://cellphones.com.vn"
LISTING_URL = f"{BASE_URL}/mobile.html"
OUTPUT_FILE = "crawl_phone_cellphones/products.json"
OUTPUT_DIR = "crawl_phone_cellphones"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    "Accept": "application/json, text/html, */*",
    "Referer": BASE_URL,
}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

# ── Tiny slug generator (database-safe) ──────────────────────────
def make_slug(text: str) -> str:
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    text = re.sub(r"[^a-z0-9\s-]", "", text.lower())
    text = re.sub(r"[\s_]+", "-", text.strip())
    return re.sub(r"-+", "-", text)[:160]

def make_sku(product_id: str, variant_idx: int, color: str, ram_gb: int = 0, storage_gb: int = 0) -> str:
    core = f"{product_id}-{variant_idx}"
    attrs = f"{color or 'std'}{ram_gb or ''}{storage_gb or ''}"
    short = hashlib.md5(attrs.encode()).hexdigest()[:8].upper()
    return f"CPS-{core}-{short}"

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

print("Configuration ready.")

Configuration ready.


In [3]:
# ─────────────────────────────────────────────
# 2. Master data: Brand + Category  (pre-loaded from DB seed)
# ─────────────────────────────────────────────
# These match the TMDT brands table — slug is the unique key used for lookups.
BRANDS = {
    slug: {"id": id_, "name": name, "slug": slug}
    for id_, name, slug in [
        (1, "Apple",      "apple"),
        (2, "Samsung",    "samsung"),
        (3, "Xiaomi",     "xiaomi"),
        (4, "OPPO",       "oppo"),
        (5, "vivo",       "vivo"),
        (6, "realme",     "realme"),
        (7, "Nokia",      "nokia"),
        (8, "ASUS",       "asus"),
        (9, "Huawei",     "huawei"),
        (10,"POCO",       "poco"),
        (11,"itel",       "itel"),
        (12,"Infinix",    "infinix"),
        (13,"Motorola",   "motorola"),
        (14,"Nothing",    "nothing"),
        (15,"Nubia",      "nubia"),
    ]
}

CATEGORIES = {
    "dien-thoai": {"id": 1, "name": "Điện thoại",    "slug": "dien-thoai"},
    "may-tinh-bang": {"id": 2, "name": "Máy tính bảng", "slug": "may-tinh-bang"},
    "laptop":     {"id": 3, "name": "Laptop",           "slug": "laptop"},
    "phu-kien":  {"id": 4, "name": "Phụ kiện",         "slug": "phu-kien"},
}

print(f"Loaded {len(BRANDS)} brands and {len(CATEGORIES)} categories.")

Loaded 15 brands and 4 categories.


In [4]:
# ─────────────────────────────────────────────
# 3. Stage 1 — Collect ALL product detail URLs
#    Uses BeautifulSoup + requests (no browser/API dependency).
# ─────────────────────────────────────────────

def _fetch_with_session(url: str) -> BeautifulSoup | None:
    """Fetch URL via SESSION, return BeautifulSoup or None."""
    try:
        r = SESSION.get(url, timeout=15)
        r.raise_for_status()
        return BeautifulSoup(r.text, "lxml")
    except Exception:
        return None


def collect_all_product_urls() -> list[str]:
    """
    Crawl all pages of /mobile.html using BeautifulSoup.
    Returns a sorted, de-duplicated list of product detail URLs.
    Matches the logic that successfully ran in scraper.ipynb (20 URLs confirmed).
    """
    print("=== Stage 1: Collecting product URLs ===")

    all_urls: set[str] = set()
    page = 1

    while page <= 100:   # safety cap
        page_url = f"{LISTING_URL}?page={page}" if page > 1 else LISTING_URL
        print(f"  Crawling listing page {page}: {page_url}", end="  ", flush=True)

        soup = _fetch_with_session(page_url)
        if not soup:
            print("❌  (fetch failed)")
            break

        # ── Product card selectors (trial-and-error for cellphones.com.vn) ──
        cards: list[Tag] = []
        for sel in [
            "a.product-item",
            ".product-list-filter a[href*='.html']",
            ".cps-product-item a",
            "a[href*='/mobile/']",
            "[class*='product'] a[href$='.html']",
            ".category-product-list a[href*='.html']",
        ]:
            cards = soup.select(sel)
            if cards:
                print(f"✅  {len(cards)} cards", end="")
                break
        else:
            print("❌  (no product cards found)")
            break

        new_found = 0
        for card in cards:
            href = card.get("href", "") or ""
            if not href:
                continue
            # Make absolute URL
            if href.startswith("/"):
                href = BASE_URL + href
            elif not href.startswith("http"):
                href = BASE_URL + "/" + href

            # Only keep product detail pages:
            # - Must end with .html
            # - Must NOT be category/subcategory pages (don't contain /mobile/ as directory)
            # - Must NOT be static pages (laptop, tablet, etc.)
            # Valid examples:
            #   https://cellphones.com.vn/iphone-17-pro.html
            #   https://cellphones.com.vn/dien-thoai-honor-400-5g-12gb-512gb.html
            # Excluded:
            #   https://cellphones.com.vn/mobile.html
            #   https://cellphones.com.vn/mobile/apple.html
            #   https://cellphones.com.vn/laptop.html
            #   https://cellphones.com.vn/thiet-bi-am-thanh.html
            if not href.endswith(".html"):
                continue
            # Skip category / subcategory pages (contain /mobile/ as directory)
            if "/mobile/" in href and "/mobile/" not in href.split(BASE_URL)[-1].split("?")[0]:
                # actually skip if it's a /mobile/ subdir page
                pass
            path = href.split(BASE_URL)[-1].split("?")[0]
            if "/mobile/" in path and not path.startswith("/mobile/"):
                # /mobile/ appears but not as top-level dir (e.g. /mobile/apple.html)
                continue
            # Skip known non-product paths
            skip_prefixes = [
                "/mobile/", "/tablet.", "/laptop.", "/man-hinh.",
                "/dong-ho.", "/am-thanh.", "/do-choi.", "/do-gia-dung.",
                "/hang-cu/", "/phu-kien/", "/thiet-bi.", "/dien-may.",
            ]
            is_product = True
            for prefix in skip_prefixes:
                if path.startswith(prefix):
                    is_product = False
                    break
            # Require path to contain a product slug indicator
            # (no top-level category pages like /mobile.html, /laptop.html)
            if is_product and path not in ("/mobile.html", "/tablet.html", "/laptop.html", "/mobile"):
                if href not in all_urls:
                    all_urls.add(href)
                    new_found += 1

        print(f"  +{new_found} new  (total: {len(all_urls)})")

        # ── Pagination: look for "Trang tiếp" / next page link ──
        next_btn = (
            soup.select_one("a.next, a[aria-label='Next'], .pagination .next a")
            or soup.select_one(f"a[href*='page={page+1}']")
            or soup.select_one("a[rel='next']")
        )
        if not next_btn:
            print("  No next-page link found — pagination ended.")
            break

        page += 1
        time.sleep(1.5)

    result = sorted(all_urls)
    print(f"\n✅ Total unique product URLs: {len(result)}")
    return result


# ── Run ──────────────────────────────────────────────────────────
all_product_urls = collect_all_product_urls()

=== Stage 1: Collecting product URLs ===
  Crawling listing page 1: https://cellphones.com.vn/mobile.html  ✅  20 cards  +20 new  (total: 20)
  No next-page link found — pagination ended.

✅ Total unique product URLs: 20


In [5]:
# ─────────────────────────────────────────────
# 4. Stage 2 — Scrape each product detail page
# ─────────────────────────────────────────────

def extract_json_from_page(html: str) -> dict | None:
    """
    Walk through multiple strategies to extract structured JSON from a page:
    1. window.__NEXT_DATA__   (Next.js SSR)
    2. window.__STATE__       (Next.js client hydration)
    3. <script type="application/json"> tags
    4. data-* attributes on root element
    """
    soup = BeautifulSoup(html, "lxml")

    # Strategy 1
    m = re.search(r'"product"\s*:\s*\{', html)
    if m:
        # Try to find the full JSON object using brace counting
        start = html.rfind('{', 0, m.start())
        depth, pos = 0, start
        for i, ch in enumerate(html[start:], start):
            if ch == '{': depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    try:
                        obj = json.loads(html[start:i+1])
                        # Return the inner product dict if wrapped
                        if "product" in obj:
                            return obj["product"]
                        if "data" in obj:
                            return obj["data"]
                        return obj
                    except json.JSONDecodeError:
                        pass
                    break

    # Strategy 2: window.__NEXT_DATA__
    m = re.search(r'window\.__NEXT_DATA__\s*=\s*({.*?})\s*;?\s*$',
                  html, re.DOTALL | re.MULTILINE)
    if m:
        try:
            nd = json.loads(m.group(1))
            page_props = nd.get("props", {}).get("pageProps", {})
            # The product is usually nested here
            for key in ("product", "data", "productData", "product_detail"):
                if key in page_props:
                    return page_props[key]
            return page_props
        except Exception:
            pass

    # Strategy 3: <script type="application/ld+json">
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            obj = json.loads(tag.string or tag.get_text())
            if isinstance(obj, dict):
                for key in ("product", "offers"):
                    if key in obj:
                        return obj
            return obj
        except Exception:
            pass

    return None


def parse_ram_storage(label: str) -> tuple[int | None, str | None]:
    """
    Parse RAM / storage label like '8GB / 256GB' or '12GB / 512GB'.
    Returns (ram_gb, storage_label).
    """
    if not label:
        return None, None
    label = label.strip()
    m = re.search(r'(\d+)\s*GB?\s*[/\s]+(\d+)\s*GB', label, re.IGNORECASE)
    if m:
        return int(m.group(1)), f"{m.group(2)}GB"
    m2 = re.search(r'(\d+)\s*GB', label, re.IGNORECASE)
    if m2:
        return int(m2.group(1)), None
    return None, label


def parse_specs_table(soup: BeautifulSoup) -> list[dict]:
    """
    Parse spec table on the detail page.
    Returns list of {spec_key, spec_value}.
    """
    specs = []
    rows = soup.select("table.specs-table tr, div.spec-row, li.spec-item")
    for row in rows:
        cells = [c.get_text(strip=True) for c in row.find_all(["td", "span", "div"])]
        if len(cells) >= 2:
            specs.append({
                "spec_key":   cells[0].rstrip(":"),
                "spec_value": cells[1],
            })
    return specs


def resolve_brand(brand_name_raw: str) -> dict | None:
    """Fuzzy-match a raw brand name to the known BRANDS dict."""
    if not brand_name_raw:
        return None
    key = brand_name_raw.lower().strip()
    if key in BRANDS:
        return BRANDS[key]
    for slug, info in BRANDS.items():
        if slug in key or key in slug:
            return info
        if info["name"].lower() in key:
            return info
    return None


def build_product_object(raw: dict, detail_html: str, product_url: str) -> dict:
    """
    Transform scraped/raw data into the TMDT Product + Variants schema.
    Returns a flat dict that maps directly to the database Product-Variant model.
    """
    soup = BeautifulSoup(detail_html, "lxml")

    # ── Basic fields ─────────────────────────────────────────────
    name          = raw.get("name") or raw.get("product_name") or ""
    slug_raw      = raw.get("slug") or raw.get("url_key") or name
    brand_info    = resolve_brand(
        raw.get("brand") or raw.get("manufacturer") or ""
    )
    category_info = CATEGORIES.get("dien-thoai")   # default mobile category

    # ── Pricing ───────────────────────────────────────────────────
    price = raw.get("price") or raw.get("price_after_discount") or raw.get("sale_price") or 0
    compare_price = raw.get("original_price") or raw.get("price_before_discount") or 0
    if isinstance(price, str):
        price = int(re.sub(r"[^\d]", "", price))
    if isinstance(compare_price, str):
        compare_price = int(re.sub(r"[^\d]", "", compare_price))

    # Sale percent
    sale_pct = 0
    if compare_price and price and compare_price > price:
        sale_pct = round((compare_price - price) / compare_price * 100)

    # ── Thumbnail ─────────────────────────────────────────────────
    thumbnail = (
        raw.get("thumbnail") or raw.get("feature_image") or
        raw.get("images", [{}])[0].get("src", "")
        if isinstance(raw.get("images"), list) else ""
    )

    # ── Description ──────────────────────────────────────────────
    short_desc = (
        raw.get("short_description") or raw.get("excerpt") or
        raw.get("summary", "") or ""
    )
    detail_desc = (
        raw.get("detail_description") or raw.get("description") or
        raw.get("content", "") or
        "\n".join(p.get_text(strip=True)
                  for p in soup.select("div.product-desc p, div.description-content p"))
    )
    if len(short_desc) > 500:
        short_desc = short_desc[:497] + "…"

    # ── Specs ────────────────────────────────────────────────────
    specs = parse_specs_table(soup)
    raw_specs = raw.get("specifications") or raw.get("specs") or []
    if isinstance(raw_specs, list):
        for s in raw_specs:
            if isinstance(s, dict):
                specs.append({
                    "spec_key":   s.get("key") or s.get("name", ""),
                    "spec_value": s.get("value") or s.get("val", ""),
                })

    # ── Images ───────────────────────────────────────────────────
    raw_imgs = raw.get("images") or raw.get("gallery") or []
    all_imgs = []
    for img in (raw_imgs if isinstance(raw_imgs, list) else []):
        src = img.get("src") or img.get("url") or img.get("image", "")
        if src:
            all_imgs.append(src)

    # Also scan page for gallery images
    for tag in soup.select('img[src*="product"], img[src*="cdn.cellphones"]'):
        src = tag.get("src") or ""
        if src and src not in all_imgs:
            all_imgs.append(src)

    # ── Variants (colour + RAM/ROM combos) ───────────────────────
    raw_variants = raw.get("variants") or raw.get("configurations") or []
    if not raw_variants:
        # Synthesise a single default variant from price/stock data
        raw_variants = [{"color": raw.get("color") or "Default", "price": price}]

    variants = []
    for i, v in enumerate(raw_variants):
        if isinstance(v, dict):
            color_name = v.get("color") or v.get("colour") or "Default"
            color_code = v.get("color_code") or v.get("hex") or ""
            var_price  = v.get("price") or price
            var_compare = v.get("compare_at_price") or v.get("original_price") or compare_price
            ram_lbl    = v.get("ram") or v.get("ram_label") or ""
            stor_lbl   = v.get("storage") or v.get("rom") or ""
            ram_gb, storage_label = parse_ram_storage(ram_lbl or stor_lbl or "")
            if ram_lbl and not ram_gb:
                ram_gb, storage_label = parse_ram_storage(ram_lbl)
            storage_gb = int(re.sub(r"[^\d]", "", storage_label)) if storage_label else None
            img_list   = v.get("images") or v.get("gallery") or []
            var_imgs   = [img.get("src") or img.get("url", "") for img in img_list
                          if img.get("src") or img.get("url", "")] or []

            sku = make_sku(
                product_id=re.sub(r"[^\w]", "-", slug_raw)[:40],
                variant_idx=i,
                color=color_code or color_name,
                ram_gb=ram_gb,
                storage_gb=storage_gb,
            )
            variants.append({
                "sku":             sku,
                "color":           color_name,
                "color_code":      color_code,
                "ram_gb":          ram_gb,
                "storage_label":   storage_label,
                "storage_gb":      storage_gb,
                "price":           float(var_price or 0),
                "compare_at_price": float(var_compare or 0) or None,
                "weight_gram":     v.get("weight") or raw.get("weight"),
                "color_image_url": var_imgs[0] if var_imgs else "",
                "images":          var_imgs,
            })

    # ── Rating / reviews ──────────────────────────────────────────
    rating       = raw.get("rating") or raw.get("average_rating") or 0
    review_count = raw.get("review_count") or raw.get("total_reviews") or 0

    # ── Assemble final product object ──────────────────────────────
    product = {
        # ── Product fields ───────────────────────────────────────
        "brand_id":             brand_info["id"] if brand_info else None,
        "brand_name":           brand_info["name"] if brand_info else (raw.get("brand") or ""),
        "brand_slug":           brand_info["slug"] if brand_info else "",
        "category_id":          category_info["id"],
        "category_name":        category_info["name"],
        "category_slug":        category_info["slug"],
        "name":                 name,
        "base_name":            re.sub(r"\s*\(.*?\)\s*$", "", name).strip(),
        "slug":                 make_slug(slug_raw),
        "short_description":    short_desc[:500],
        "detail_description":   detail_desc,
        "thumbnail_url":        thumbnail,
        "warranty_months":      raw.get("warranty") or raw.get("warranty_months") or 12,
        "sale":                 sale_pct,
        "status":               "ACTIVE" if raw.get("in_stock", True) else "INACTIVE",
        "is_featured":          raw.get("is_featured") or raw.get("featured") or False,
        "product_url":          product_url,

        # ── Nested Variants ───────────────────────────────────────
        "variants": variants,

        # ── Related tables ────────────────────────────────────────
        "images":         all_imgs,
        "specifications": specs,
        "rating":         float(rating) if rating else 0.0,
        "review_count":   int(re.sub(r"[^\d]", "", str(review_count))) if review_count else 0,
        "scraped_at":     now_iso(),
    }
    return product


def scrape_product_page(url: str) -> dict | None:
    """
    Fetch a product detail page and extract structured JSON.
    Returns the mapped product object or None on failure.
    """
    try:
        r = SESSION.get(url, timeout=20)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        html = r.text
    except Exception as exc:
        print(f"  ⚠ Network error fetching {url}: {exc}")
        return None

    # Extract structured JSON from page
    raw = extract_json_from_page(html)
    if not raw:
        # Save raw HTML for manual inspection
        print(f"  ⚠ No structured JSON found in {url}")
        return None

    return build_product_object(raw, html, url)


def scrape_all_products(urls: list[str], delay: float = 1.5) -> list[dict]:
    """
    Iterate over all product URLs, scrape details, and return the list.
    Skips already-saved products if resuming.
    """
    results = []
    errors  = []

    # Try to resume from partial output
    try:
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            existing = json.load(f)
        results = [p for p in existing if p.get("slug")]
        print(f"📂 Resuming: {len(results)} products already saved.")
    except FileNotFoundError:
        existing = []

    seen_slugs = {p["slug"] for p in results}

    for i, url in enumerate(urls, 1):
        if url in seen_slugs:
            print(f"[{i}/{len(urls)}] Skip (already scraped): {url}")
            continue

        slug = url.rstrip("/").split("/")[-1]
        if slug in seen_slugs:
            print(f"[{i}/{len(urls)}] Skip (already scraped): {slug}")
            continue

        print(f"[{i}/{len(urls)}] Scraping: {url}", end="  ", flush=True)
        product = scrape_product_page(url)

        if product:
            results.append(product)
            seen_slugs.add(product["slug"])
            print(f"✅  {len(product['variants'])} variants")
        else:
            errors.append(url)
            print("❌")

        # Checkpoint every 25 products
        if i % 25 == 0:
            with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"\n💾 Checkpoint saved ({len(results)} products)")

        time.sleep(delay)

    return results, errors


print("Stage 2 functions defined.")

Stage 2 functions defined.


In [6]:
# ─────────────────────────────────────────────
# 5. Run Stage 2 (or skip if URL list is empty)
# ─────────────────────────────────────────────
try:
    urls_to_scrape = all_product_urls
except NameError:
    urls_to_scrape = []

if not urls_to_scrape:
    print("⚠ No URLs collected — check Stage 1.")
else:
    print(f"\n=== Stage 2: Scraping {len(urls_to_scrape)} product pages ===")
    final_products, failed_urls = scrape_all_products(urls_to_scrape, delay=1.5)

    print(f"\n✅ Scraping complete: {len(final_products)} products, {len(failed_urls)} failures")
    if failed_urls:
        print("Failed URLs:")
        for u in failed_urls:
            print(f"  • {u}")


=== Stage 2: Scraping 20 product pages ===
[1/20] Scraping: https://cellphones.com.vn/dien-thoai-itel-p55-plus-8gb-256gb.html  ✅  1 variants
[2/20] Scraping: https://cellphones.com.vn/dien-thoai-nubia-neo-5-5g.html  ✅  1 variants
[3/20] Scraping: https://cellphones.com.vn/dien-thoai-nubia-neo-5-gt.html  ✅  1 variants
[4/20] Scraping: https://cellphones.com.vn/dien-thoai-oppo-find-n6.html  ✅  1 variants
[5/20] Scraping: https://cellphones.com.vn/dien-thoai-oppo-find-x9s.html  ✅  1 variants
[6/20] Scraping: https://cellphones.com.vn/dien-thoai-oppo-reno15-f.html  ✅  1 variants
[7/20] Scraping: https://cellphones.com.vn/dien-thoai-samsung-galaxy-a06-5g.html  ✅  1 variants
[8/20] Scraping: https://cellphones.com.vn/dien-thoai-samsung-galaxy-a07.html  ✅  1 variants
[9/20] Scraping: https://cellphones.com.vn/dien-thoai-samsung-galaxy-a17-5g.html  ✅  1 variants
[10/20] Scraping: https://cellphones.com.vn/dien-thoai-samsung-galaxy-a57.html  ✅  1 variants
[11/20] Scraping: https://cellphones.c

In [7]:
# ─────────────────────────────────────────────
# 6. Save final JSON & DB-ready export utilities
# ─────────────────────────────────────────────

def save_final_json(products: list[dict]):
    """Write the complete product list to the output JSON file."""
    import os
    os.makedirs(os.path.dirname(OUTPUT_FILE) or ".", exist_ok=True)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(products, f, ensure_ascii=False, indent=2)
    print(f"💾 Saved {len(products)} products → {OUTPUT_FILE}")


def export_for_db_import(products: list[dict], out_dir: str | None = None):
    """
    Split products into separate JSON files mirroring each DB table.
    Ready to bulk-import or POST to REST API.
    """
    import os
    out_dir = out_dir or OUTPUT_DIR
    os.makedirs(out_dir, exist_ok=True)

    brands_out, categories_out = [], []
    products_out, variants_out = [], []
    images_out, specs_out = [], []
    inventories_out = []

    for p in products:
        bid  = p["brand_id"] or 0
        cid  = p["category_id"] or 1

        # Brand
        if p["brand_name"]:
            brands_out.append({
                "id": bid, "name": p["brand_name"],
                "slug": p["brand_slug"] or make_slug(p["brand_name"]),
                "is_active": True,
            })

        # Category
        if p["category_name"]:
            categories_out.append({
                "id": cid, "name": p["category_name"],
                "slug": p.get("category_slug") or make_slug(p["category_name"]),
                "is_active": True,
            })

        # Product
        products_out.append({
            "brand_id":          bid,
            "category_id":        cid,
            "name":              p["name"],
            "base_name":         p.get("base_name", p["name"]),
            "slug":              p["slug"],
            "short_description": p.get("short_description", ""),
            "detail_description": p.get("detail_description", ""),
            "thumbnail_url":     p.get("thumbnail_url", ""),
            "warranty_months":   p.get("warranty_months", 12),
            "sale":              p.get("sale", 0),
            "status":            p.get("status", "ACTIVE"),
            "is_featured":       p.get("is_featured", False),
        })

        # Variants
        pid = len(products_out)
        for v in p.get("variants", []):
            v["product_id"] = pid
            variants_out.append({
                "product_id":       pid,
                "sku":              v["sku"],
                "color":            v.get("color", "Default"),
                "ram_gb":           v.get("ram_gb"),
                "storage_label":    v.get("storage_label"),
                "storage_gb":       v.get("storage_gb"),
                "price":            v.get("price", 0),
                "compare_at_price": v.get("compare_at_price"),
                "weight_gram":      v.get("weight_gram"),
                "is_active":        True,
            })
            vid = len(variants_out)
            # ProductImage
            for rank, img_url in enumerate(v.get("images", []) or []):
                images_out.append({
                    "product_id": pid, "variant_id": vid,
                    "image_url": img_url,
                    "is_primary": rank == 0,
                    "sort_order": rank,
                })
            # Inventory (default IN_STOCK)
            inventories_out.append({
                "variant_id":        vid,
                "quantity_on_hand":  10,
                "stock_status":      "IN_STOCK",
            })

        # ProductImage (extra gallery images at product level)
        for rank, img_url in enumerate(p.get("images", []) or []):
            if not any(x["image_url"] == img_url for x in images_out[-len(p.get("variants", []))*5:]):
                images_out.append({
                    "product_id": pid, "variant_id": None,
                    "image_url": img_url,
                    "is_primary": False,
                    "sort_order": rank,
                })

        # ProductSpecification
        for rank, spec in enumerate(p.get("specifications", []) or []):
            specs_out.append({
                "product_id": pid,
                "spec_key":   spec.get("spec_key", ""),
                "spec_value": spec.get("spec_value", ""),
                "sort_order": rank,
            })

    # De-duplicate brands & categories
    brands_out = {b["id"]: b for b in brands_out if b["id"]}.values()
    categories_out = {c["id"]: c for c in categories_out if c["id"]}.values()

    tables = {
        "brands.json":                list(brands_out),
        "categories.json":           list(categories_out),
        "products.json":             products_out,
        "product_variants.json":     variants_out,
        "product_images.json":       images_out,
        "product_specifications.json": specs_out,
        "inventories.json":          inventories_out,
    }

    for fname, data in tables.items():
        path = os.path.join(out_dir, fname)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"  • {fname}: {len(data)} rows")

    print(f"\n✅ DB-ready export complete → {out_dir}/")


# ── Run final save ──────────────────────────────────────────────
if "final_products" in dir() and final_products:
    save_final_json(final_products)
    export_for_db_import(final_products)
else:
    print("⚠ No final_products found — run cells 4 → 6 first.")
    print("   Trying to load from checkpoint file …")
    try:
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            cached = json.load(f)
        print(f"   Loaded {len(cached)} products from checkpoint.")
        save_final_json(cached)
        export_for_db_import(cached)
    except FileNotFoundError:
        print("   No checkpoint found. Run the full pipeline.")

💾 Saved 20 products → crawl_phone_cellphones/products.json
  • brands.json: 0 rows
  • categories.json: 1 rows
  • products.json: 20 rows
  • product_variants.json: 20 rows
  • product_images.json: 88 rows
  • product_specifications.json: 0 rows
  • inventories.json: 20 rows

✅ DB-ready export complete → crawl_phone_cellphones/


In [8]:
# ─────────────────────────────────────────────
# 7. Data-quality report & schema validation
# ─────────────────────────────────────────────

def validate_and_report(filepath: str = OUTPUT_FILE):
    """Print a summary report of scraped data vs DB schema."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            products = json.load(f)
    except FileNotFoundError:
        print("No products file found.")
        return

    total_products  = len(products)
    total_variants  = sum(len(p.get("variants", [])) for p in products)
    total_images    = sum(len(p.get("images", [])) for p in products)
    total_specs     = sum(len(p.get("specifications", [])) for p in products)
    brands_found    = set(p.get("brand_name", "") for p in products if p.get("brand_name"))
    avg_price       = []
    sale_products   = [p for p in products if p.get("sale", 0) > 0]
    featured        = [p for p in products if p.get("is_featured")]
    no_brand        = [p for p in products if not p.get("brand_id")]
    no_variants     = [p for p in products if not p.get("variants")]

    for p in products:
        for v in p.get("variants", []):
            if v.get("price"):
                avg_price.append(v["price"])

    print("=" * 60)
    print("  CELLPHONES SCRAPE REPORT — TMDT Schema Validation")
    print("=" * 60)
    print(f"  Products scraped   : {total_products}")
    print(f"  Total variants     : {total_variants}")
    print(f"  Total images       : {total_images}")
    print(f"  Total specs        : {total_specs}")
    print(f"  Brands found       : {len(brands_found)}  {sorted(brands_found)}")
    print(f"  Products on sale   : {len(sale_products)}")
    print(f"  Featured products  : {len(featured)}")
    print(f"  No brand_id        : {len(no_brand)} ⚠️")
    print(f"  No variants        : {len(no_variants)} ⚠️")
    if avg_price:
        print(f"  Avg variant price  : {sum(avg_price)/len(avg_price):,.0f} VND")
    print("=" * 60)

    # DB-schema column coverage check
    required_product_cols = [
        "brand_id", "category_id", "name", "base_name", "slug",
        "short_description", "detail_description", "thumbnail_url",
        "warranty_months", "sale", "status", "is_featured",
    ]
    required_variant_cols = [
        "sku", "color", "ram_gb", "storage_label", "storage_gb",
        "price", "compare_at_price", "weight_gram", "is_active",
    ]
    if products:
        sample = products[0]
        print("\n  Product fields coverage:")
        for col in required_product_cols:
            present = col in sample
            print(f"    {'✅' if present else '❌'}  {col}")
        print("\n  Variant fields coverage (from first product):")
        for v in (sample.get("variants") or [{}]):
            for col in required_variant_cols:
                present = col in v
                print(f"    {'✅' if present else '❌'}  {col}")
            break
    print()


validate_and_report()

  CELLPHONES SCRAPE REPORT — TMDT Schema Validation
  Products scraped   : 20
  Total variants     : 0
  Total images       : 0
  Total specs        : 0
  Brands found       : 0  []
  Products on sale   : 0
  Featured products  : 0
  No brand_id        : 20 ⚠️
  No variants        : 20 ⚠️

  Product fields coverage:
    ✅  brand_id
    ✅  category_id
    ✅  name
    ✅  base_name
    ✅  slug
    ✅  short_description
    ✅  detail_description
    ✅  thumbnail_url
    ✅  warranty_months
    ✅  sale
    ✅  status
    ✅  is_featured

  Variant fields coverage (from first product):
    ❌  sku
    ❌  color
    ❌  ram_gb
    ❌  storage_label
    ❌  storage_gb
    ❌  price
    ❌  compare_at_price
    ❌  weight_gram
    ❌  is_active

